# Real-time Differentiable Multi-Layer Perceptron (RtdMLP)
### Problem
Sometimes, it will be helpful to have the Jacobian of a learned model in real-time. For example, we might have a model that predicts the control points given coil currents.

At time of writing it doesn't seem real-time automatic differentiation is really a thing yet.

However, it is simple enough to write the analytic Jacobian of a multi-layer perceptron (MLP) to run in real-time.

### Derivation
The ith layer of a MLP with activation $\sigma$ is given by:

$$
\mathbf{z}_i = \mathbf{W}_i \mathbf{h}_{i-1} + \mathbf{b}_i\\
\mathbf{h}_i = \sigma(\mathbf{z}_i)
$$

where $\mathbf{h}_0$ is the network input, $\mathbf{h}_L$ is the output, $\mathbf{h}_i$ for all other i are hidden states, $\mathbf{W}_i$ is the weight matrix, and $\mathbf{b}_i$ is the bias vector. From this, we can see that the layer-wise Jacobian is given by:
$$
\frac{\partial \mathbf{h}_i}{\partial \mathbf{h}_{i-1}} = \text{diag}\left(\frac{\partial\sigma}{\partial \mathbf{z}_i}(\mathbf{z}_i)\right) \mathbf{W}_i
$$
and thus the Jacobian of the entire network is given by:
$$
\frac{\partial \mathbf{h}_L}{\partial \mathbf{h}_0} = \prod_{i=1}^L \frac{\partial \mathbf{h}_i}{\partial \mathbf{h}_{i-1}}
$$

In [ ]:
import jax
import jax.numpy as jnp

from popsim.ml.rtd_mlp import Activation, RtdMLP

mlp = RtdMLP(
    in_size=2,
    out_size=2,
    width_size=32,
    depth=2,
    activation=Activation.RELU,
    final_activation=Activation.SOFTMAX,
    key=jax.random.PRNGKey(0),
)

x = jax.random.normal(jax.random.PRNGKey(0), (2,))

y, y_jac = mlp(x, return_jacobian=True)

# Compute the autodiff Jacobian
y_jac_autodiff = jax.jacfwd(mlp)(x)

print("Prediction:")
print(y)
print("Jacobian:")
print(y_jac)
print("Jacobian (autodiff):")
print(y_jac_autodiff)
assert jnp.allclose(y_jac, y_jac_autodiff), "Jacobians do not match!"